# Notebook 2: Embedding, storage and a first search

**Goal:** run the real ingestion pipeline on your documents (extract, chunk, embed, store) and see the first retrieval results with similarity scores.

**Before you start**
- Docker Desktop is running and the database is up (`docker compose up -d`).
- `.env` contains your real `VOYAGE_API_KEY`. Embedding a few hundred pages costs a few cents at most.
- Your documents are in `data/sample_docs/`.
- The `.venv` kernel is selected.

In [5]:
import sys
from pathlib import Path

# Make the project importable when the notebook runs from the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.config import settings
from app.db import connection
from app.ingestion import store
from app.ingestion.embed import embed_query
from app.ingestion.errors import DuplicateDocumentError
from app.ingestion.pipeline import ingest_file

USER_ID = 1  # the development user created by schema.sql
DOCS = ROOT / "data" / "sample_docs"
files = sorted(p for p in DOCS.iterdir() if p.suffix.lower() in settings.allowed_extensions)
print("Voyage key loaded:", bool(settings.voyage_api_key))  # never print the key itself
print("Files to ingest:", [f.name for f in files])

Voyage key loaded: True
Files to ingest: ['Deputy_Demographer_BPS17_Complete_Preparation_Guide.pdf', 'Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx']


## 1. Ingest
Each file goes through: type check, duplicate check (file hash), extraction, chunking, embedding and storage. Running this cell twice is safe: files already stored are skipped.

In [18]:
for path in files:
    try:
        record = await ingest_file(path, path.name, USER_ID)
        print(f"ingested  {record.filename}: {record.chunk_count} chunks (document id {record.id})")
    except DuplicateDocumentError as exc:
        print(f"skipped   {path.name}: {exc}")
    except Exception as exc:  # show the reason, then continue with the next file
        print(f"FAILED    {path.name}: {type(exc).__name__}: {exc}")


=== Document parser messages ===
ingested  Deputy_Demographer_BPS17_Complete_Preparation_Guide.pdf: 118 chunks (document id 30)
skipped   Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx: This document was already uploaded (id 9).


## 2. What is stored

In [19]:
for doc in store.list_documents(USER_ID):
    print(f"id {doc.id:>3} | {doc.filename:<40} | pages {doc.page_count} | chunks {doc.chunk_count}")

id  30 | Deputy_Demographer_BPS17_Complete_Preparation_Guide.pdf | pages 77 | chunks 118
id   9 | Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx | pages None | chunks 17


## 3. First search
The simplest possible retrieval: embed the question, then find the chunks with the highest cosine similarity. Real retrieval (hybrid search and reranking) comes next. **Edit the first two questions** so your documents can answer them. The third is unanswerable on purpose.

In [21]:
async def search(question: str, k: int = 5) -> None:
    """Print the k chunks closest to the question, with cosine similarity."""
    vector = store.to_vector_literal(await embed_query(question))
    with connection() as conn:
        rows = conn.execute(
            """
            SELECT d.filename, c.heading, c.page_start, c.page_end,
                   1 - (c.embedding <=> %s::vector) AS similarity, c.content
            FROM chunks c JOIN documents d ON d.id = c.document_id
            WHERE c.user_id = %s
            ORDER BY c.embedding <=> %s::vector
            LIMIT %s
            """,
            (vector, USER_ID, vector, k),
        ).fetchall()
    print(f"\nQUESTION: {question}")
    for rank, (filename, heading, first, last, similarity, content) in enumerate(rows, start=1):
        if first is None:
            where = "no pages"
        else:
            where = f"p.{first}" if first == last else f"p.{first}-{last}"
        print(f"  {rank}. score {similarity:.3f} | {filename} | {where} | {heading}")
        print(f"     {content[:160]!r}")


QUESTIONS = [
    "REPLACE WITH A QUESTION YOUR DOCUMENT CAN ANSWER",
    "REPLACE WITH A SECOND QUESTION",
    "What is the capital of Mars?",  # unanswerable on purpose
]
for question in QUESTIONS:
    await search(question)


QUESTION: REPLACE WITH A QUESTION YOUR DOCUMENT CAN ANSWER
  1. score 0.398 | Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx | no pages | Phase 8 — Generate answers from retrieved context > Step 15: Create a grounded answer function
     'Create answer.py. The function should receive the question and retrieved passages. Build a prompt that explicitly says to answer only from context, cite sources'
  2. score 0.381 | Deputy_Demographer_BPS17_Complete_Preparation_Guide.pdf | p.75 | COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE > Chapter 15. Short Topic Quizzes and Self-Assessment
     'Use these mini-quizzes after each chapter. Cover the answer until you have attempted the question.\n\n_Demography_\n\nDefine natural increase and net migration. Exp'
  3. score 0.369 | Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx | no pages | Phase 9 — Connect upload and chat flow > Step 16: Wire the pipeline into Streamlit
     'When files are uploaded, validate and extract text.\n\nCh

### Read the scores
For questions your documents answer, the top score should be clearly higher than for the nonsense question. Write down the typical values: they help choose the similarity threshold the service uses to refuse to answer.

## 4. Delete test
Put a document id from the list above in `DELETE_ID` and run the cell. The document's chunks and vectors are removed, and the search no longer finds its content.

In [23]:
DELETE_ID = None  # for example: DELETE_ID = 3
if DELETE_ID is not None:
    print("deleted:", store.delete_document(USER_ID, DELETE_ID))
    print("documents left:", [d.filename for d in store.list_documents(USER_ID)])
    await search(QUESTIONS[0])
else:
    print("Set DELETE_ID first.")

Set DELETE_ID first.


## Notes for DESIGN.md (fill these in)
- Typical top score for answerable questions vs. the nonsense question.
- Did the right chunk rank first? If not, what ranked above it?
- How long did ingestion take, and how many chunks per document?